In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import random, re
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from msnap.utils import common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils
from msnap.utils.container_utils import chunks
from msnap.core import msnap_prompts

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/msnap'
data_dir = f'{work_dir}/data'

# zero_shot_target = 'fact'
zero_shot_target = 'other'

in_dir = f'{data_dir}/check_contexts/zero_shot_{zero_shot_target}'
out_dir = f'{data_dir}/check_targets/zero_shot_{zero_shot_target}'

In [ ]:
model_name = 'Llama-3.2-3B'
# model_name = 'Llama-3.1-8B'

model_name_or_path = f'meta-llama/{model_name}-Instruct'
dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

model: AutoModelForCausalLM = model_utils.get_model(model_name_or_path, dtype, device, is_eval=True)

# 평가/추론 시에는 반드시 'left' 패딩
tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')

In [ ]:
in_file_path = f'{in_dir}/msnap_gpt-5.2_checked_contexts_{model_name}.json'
datas = json_utils.load_json(in_file_path)

print(json_utils.to_str(datas[0]))

In [ ]:
def add_data(datas: list, id, query, prompt, answer_fact, answer_counter):
    data = {
        'id': id,
        'query': query,
        'prompt': prompt,
        'answer_fact': answer_fact,
        'answer_counter': answer_counter
    }

    datas.append(data)

In [ ]:
def make_datas_poss(datas, extract_fact_size):
    datas_front, datas_back, datas_shuffle = [], [], []
    cnt_counter, cnt_counter_all = 0, 0

    for data in datas:
        id = data['id']
        query = data['question']
        answer_fact = data['answer_fact']
        answer_counter = data['answer_counter']
        context_fact_dict: dict = data['contexts_fact']
        context_counter_dict: dict = data['contexts_counter']

        contexts_fact = list(context_fact_dict.values())
        contexts_counter = list(context_counter_dict.values())

        fact_size = len(contexts_fact)
        cnt_counter_all += len(contexts_counter)

        if fact_size < extract_fact_size:
            continue

        cnt_counter += len(contexts_counter)

        for context_counter in contexts_counter:
            contexts_fact_extract = random.sample(contexts_fact, extract_fact_size)

            list_front = [context_counter] + contexts_fact_extract
            list_back = contexts_fact_extract + [context_counter]
            list_shuffle = contexts_fact_extract + [context_counter]
            random.shuffle(list_shuffle)

            prompt_front = msnap_prompts.get_generate_prompt(query, list_front)
            prompt_back = msnap_prompts.get_generate_prompt(query, list_back)
            prompt_shuffle = msnap_prompts.get_generate_prompt(query, list_shuffle)

            add_data(datas_front, id, query, prompt_front, answer_fact, answer_counter)
            add_data(datas_back, id, query, prompt_back, answer_fact, answer_counter)
            add_data(datas_shuffle, id, query, prompt_shuffle, answer_fact, answer_counter)



    print(f'make_datas_poss() extract_fact_size : {extract_fact_size}')
    print(f'make_datas_poss() cnt_counter : {cnt_counter}/{cnt_counter_all} ({(cnt_counter/cnt_counter_all):.4f})\n')

    print(f'make_datas_poss() datas_front size : {len(datas_front)}')
    print(f'make_datas_poss() datas_back size : {len(datas_back)}')
    print(f'make_datas_poss() datas_shuffle size : {len(datas_shuffle)}\n')

    return datas_front, datas_back, datas_shuffle

In [ ]:
# datas_front, datas_back, datas_shuffle = make_datas_poss(datas, 3)

# print(f'datas_front :\n{datas_front[0]}\n\n')
# print(f'datas_back :\n{datas_back[0]}\n\n')
# print(f'datas_shuffle :\n{datas_shuffle[0]}\n\n')

In [ ]:
def check_targets(datas, batch_size, out_prefix):
    checked_fact, checked_counter, checked_other = [], [], []

    for datas_batch in chunks(datas, batch_size):
        prompts = [data['prompt'] for data in datas_batch]

        generated_texts = model_utils.get_generated_texts(
            model, tokenizer, device,
            prompts, max_seq_length, max_new_tokens
        )

        for data, generated_text in zip(datas_batch, generated_texts):
            answer_fact = data['answer_fact']
            answer_counter = data['answer_counter']
            data['generated_text'] = generated_text

            if model_utils.is_correct(generated_text, answer_fact)[1]:
                checked_fact.append(data)
            elif model_utils.is_correct(generated_text, answer_counter)[1]:
                checked_counter.append(data)
            else:
                checked_other.append(data)



    print(f'check_targets() data size : {len(datas)}\n')

    cnt_fact, cnt_counter, cnt_other = len(checked_fact), len(checked_counter), len(checked_other)
    cnt_all = cnt_fact + cnt_counter + cnt_other

    print(f'check_targets cnt_fact : {cnt_fact}/{cnt_all} ({(cnt_fact/cnt_all):.4f})')
    print(f'check_targets cnt_counter : {cnt_counter}/{cnt_all} ({(cnt_counter/cnt_all):.4f})')
    print(f'check_targets cnt_other : {cnt_other}/{cnt_all} ({(cnt_other/cnt_all):.4f})\n')

    json_utils.write_json(checked_fact, f'{out_prefix}_A-fact')
    json_utils.write_json(checked_counter, f'{out_prefix}_A-counter')
    json_utils.write_json(checked_other, f'{out_prefix}_A-other')
    print()

In [ ]:
# batch_size = 2

for extract_fact_size in range(1, msnap_prompts.CONTEXT_SIZE+1):
    if extract_fact_size <= 5:
        batch_size = 5
    else:
        batch_size = 2

    datas_front, datas_back, datas_shuffle = make_datas_poss(datas, extract_fact_size)

    out_prefix = f'{out_dir}/{model_name}/maked/'
    file_prefix = f'msnap_gpt-5.2_maked_targets_{model_name}_F-{extract_fact_size}'

    json_utils.write_json(datas_front, f'{out_prefix}/{file_prefix}_C-front.json')
    json_utils.write_json(datas_back, f'{out_prefix}/{file_prefix}_C-back.json')
    json_utils.write_json(datas_shuffle, f'{out_prefix}/{file_prefix}_C-shuffle.json')
    print()

    out_prefix = f'{out_dir}/{model_name}/checked/extract_fact_{extract_fact_size}'
    file_prefix = f'msnap_gpt-5.2_checked_targets_{model_name}_F-{extract_fact_size}'

    check_targets(datas_front, batch_size, f'{out_prefix}/{file_prefix}_C-front')
    check_targets(datas_back, batch_size, f'{out_prefix}/{file_prefix}_C-back')
    check_targets(datas_shuffle, batch_size, f'{out_prefix}/{file_prefix}_C-shuffle')